# 🚀 DataMate Agent - Google Colab Training & Evaluation Pipeline
### Zalo AI Challenge: Vietnamese Offline Tabular QA System

Notebook này thiết lập môi trường GPU, tự động tải dữ liệu từ Google Drive, chạy vòng lặp **Iterative Self-Training (QLoRA + Rejection Sampling)** và đánh giá điểm số **Penalized Accuracy (PA)**.

## 1. Kiểm tra Phần cứng GPU (Khuyến nghị T4, A100 hoặc L4)

In [ ]:
!nvidia-smi

## 2. Clone Repository từ GitHub

In [ ]:
# Clone mã nguồn mới nhất từ repo của bạn
!rm -rf /content/datamate_zalo_ai
!git clone https://github.com/tuannh1905/datamate_zalo_ai.git /content/datamate_zalo_ai
%cd /content/datamate_zalo_ai/datamate_agent

## 3. Cài đặt Môi trường (vLLM, DuckDB, Unsloth, HuggingFace Base Weights)

In [ ]:
# Cài đặt dependencies và tải weights offline Qwen2.5-Coder-3B-Instruct
!bash scripts/setup_colab.sh

## 4. Tải và Chuẩn hóa Dữ liệu Cuộc thi (Hỗ trợ chạy trên NHIỀU tài khoản Colab không cần Mount Drive)

> Khi chạy Colab trên nhiều tài khoản/email khác nhau để tận dụng GPU miễn phí, việc mount Google Drive cá nhân của tài khoản khác rất bất tiện vì quyền truy cập. 
> **Giải pháp:** Chỉ cần lấy đường link chia sẻ (Google Drive link dạng 'Bất kỳ ai có liên kết đều có thể xem', link trực tiếp, hoặc Hugging Face Hub), script sẽ tự động tải, giải nén và chuẩn hóa dữ liệu trên bất kỳ tài khoản nào mà KHÔNG cần `drive.mount`!

### Cách 1 (Khuyên dùng khi chạy nhiều mail): Tải trực tiếp bằng Link URL (Không cần Mount Drive)
Dán đường link file zip của bạn vào dưới đây (hỗ trợ link Google Drive, Dropbox, link trực tiếp, v.v.):

In [ ]:
!pip install -q --upgrade gdown

# Dán link Google Drive (hoặc direct link) của file dataset zip vào đây:
# Ví dụ nếu bạn có 1 file nén chung (hoặc 2 link riêng cho train.zip và public_test.zip):
!python scripts/download_dataset.py \
    --url "https://drive.google.com/file/d/THAY_BANG_FILE_ID_CUA_BAN/view?usp=sharing"

### Cách 2 (Siêu tốc 10s cho nhiều tài khoản): Tải từ Hugging Face Dataset Hub
Tạo 1 repo dataset trên Hugging Face (ví dụ `username/zalo_datamate`), tải siêu tốc 100MB/s không lo rate limit:

In [ ]:
# !python scripts/download_dataset.py --hf_repo "username/zalo_datamate"

### Cách 3: Mount Google Drive (Nếu đang dùng chính tài khoản chứa data)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !python scripts/download_dataset.py --mode auto

## 5. Chạy Thử Nghiệm Nhanh (Smoke Test trên 5 câu hỏi)

In [ ]:
# Chạy thử suy luận và chấm điểm nhanh 5 câu hỏi để xác nhận luồng chạy trơn tru
!python scripts/run_eval.py \
    --dataset train \
    --drive_base_path /content/dataset_gdrive \
    --run_inference \
    --limit 5 \
    --verbose

## 6. Chạy Vòng lặp Tự huấn luyện (Iterative Self-Training Loop)
Quy trình tự học 3 chu kỳ:
- **Vòng 1:** Model Base sinh 12 mẫu $\to$ Rejection Sampling chọn đáp án đúng $\to$ Huấn luyện LoRA Adapter `v1`.
- **Vòng 2:** Model + Adapter `v1` sinh mẫu $\to$ Rejection Sampling $\to$ Huấn luyện LoRA Adapter `v2`.
- **Vòng 3:** Model + Adapter `v2` sinh mẫu $\to$ Rejection Sampling $\to$ Huấn luyện Adapter `v3` hoàn chỉnh.

In [ ]:
!DRIVE_BASE_PATH="/content/dataset_gdrive" \
 DATASET="train" \
 bash scripts/run_self_training.sh

## 7. Đánh giá Toàn diện với Adapter v3 và Đo lường Penalized Accuracy

In [ ]:
!python scripts/run_eval.py \
    --dataset train \
    --drive_base_path /content/dataset_gdrive \
    --adapter ./weights/adapters/v3 \
    --run_inference

## 8. Sao lưu Adapter Weights vào Google Drive cá nhân

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ZaloAI_DataMate_Backup
!cp -r ./weights/adapters /content/drive/MyDrive/ZaloAI_DataMate_Backup/
print("✓ Đã sao lưu adapter weights thành công vào Google Drive!")